In [1]:
import argparse
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from RDDPM import RDDIM
from classes import LongitudinalMRIDataset
from utils import sample_and_save

In [ ]:
sequences = ['FLAIR', 'POST', 'PRE', 'T2']
BASE_DIM = 256
GRU_LAYERS = 4
RES_BLOCKS = 2
T = 1000
ETA = 0.0
BETA_SCHEDULE = 'linear'
EPOCHS = 100
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
data_path = "training_data/"
checkpoint_dir = Path("./checkpoints")
output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)
checkpoint_dir.mkdir(exist_ok=True)
slice_axis = 2
slice_idx = 128
subsample_visits = True
min_visits = 3
log_every = 5
save_every = 10
H = 512
W = 512

In [3]:
df = pd.read_excel("training_data/clinical_data.xlsx", sheet_name=None)
most_common_id = df["Clinical_data"]['patient_id'].value_counts().idxmax()
iap = df["image_acquisition_parameters"]
acq = df["Acquisition_data"]

iap_filtered = iap[iap['patient_id'] == most_common_id]
acq_filtered = acq[acq['patient_id'] == most_common_id]

df = iap_filtered.merge(acq_filtered, on="patient_id", how="inner")

df = df[df['study_datetime_x'] == df['study_datetime_y']].copy()
df = df.drop_duplicates(subset='file_name')

date_str = df['study_datetime_x'].astype(str).str[:10]
df['file_name'] = (most_common_id + '/' + date_str + '/' + df['file_name'])

session_files = (
    df.groupby(['study_datetime_x', 'sequence_class'])['file_name']
    .first()
    .unstack(fill_value=None)
    .reset_index()
    .rename(columns={'study_datetime_x': 'session_date'})
)
session_table = session_files.sort_values('session_date').reset_index(drop=True)
print(f"Found {len(session_table)} sessions after filtering.")
print(session_table[['session_date'] + [s for s in ['PRE','POST','T2','FLAIR']
                                             if s in session_table.columns]].to_string())

Found 66 sessions after filtering.
sequence_class         session_date                                                                        PRE                                                                        POST                                                                        T2                                                                        FLAIR
0               2011-01-15_15-25-01  YG_C9D2TEGNY08A/2011-01-15/YG_C9D2TEGNY08A_2011-01-15_15-25-01_PRE.nii.gz  YG_C9D2TEGNY08A/2011-01-15/YG_C9D2TEGNY08A_2011-01-15_15-25-01_POST.nii.gz  YG_C9D2TEGNY08A/2011-01-15/YG_C9D2TEGNY08A_2011-01-15_15-25-01_T2.nii.gz  YG_C9D2TEGNY08A/2011-01-15/YG_C9D2TEGNY08A_2011-01-15_15-25-01_FLAIR.nii.gz
1               2011-02-27_08-20-07  YG_C9D2TEGNY08A/2011-02-27/YG_C9D2TEGNY08A_2011-02-27_08-20-07_PRE.nii.gz  YG_C9D2TEGNY08A/2011-02-27/YG_C9D2TEGNY08A_2011-02-27_08-20-07_POST.nii.gz  YG_C9D2TEGNY08A/2011-02-27/YG_C9D2TEGNY08A_2011-02-27_08-20-07_T2.nii.gz  YG_C9D2TEGNY08A/2011-02-27/

In [4]:
dataset = LongitudinalMRIDataset(
    session_table=session_table,
    data_root=Path(data_path),
    sequences=sequences,
    slice_axis=slice_axis,
    slice_idx=slice_idx,
    target_hw=(H, W),
    min_sessions=4
)

print(f"\nDataset initialized with {len(dataset)} sequences.")

frames, lt_seq = dataset()
n_visits = len(frames)
print(f"\nSequence length: {n_visits} visits")
print(f"Frame shape:     {frames[0].shape}  (1, C, H, W)")

Dataset: 66 sessions, 4 channels (FLAIR, POST, PRE, T2)

Dataset initialized with 66 sequences.

Sequence length: 66 visits
Frame shape:     torch.Size([1, 4, 256, 256])  (1, C, H, W)


In [ ]:
model = RDDIM(
    input_size=(H, W),
    n_channels=len(sequences),
    base_dim=BASE_DIM,
    gru_n_layers=GRU_LAYERS,
    n_res_blocks=RES_BLOCKS,
    T=T,
    eta=ETA,
    beta_schedule=BETA_SCHEDULE,
)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")
start_epoch = 0
best_loss = float('inf')

if Path(checkpoint_dir / "latest.pth").exists():
    ckpt_path = Path(checkpoint_dir + "/latest.pth")
    print(f"Resuming from {ckpt_path}")
    ckpt = torch.load(ckpt_path)
    model.load_state_dict(ckpt['model_state'])
    start_epoch = ckpt['epoch'] + 1
    best_loss = ckpt.get('best_loss', float('inf'))
    print(f"  → Resuming at epoch {start_epoch}, best loss {best_loss:.4f}")

loss_history = []
total_start = time.perf_counter()

print(f"\nTraining for {EPOCHS} epochs...")
print(f"  Sequences used : {sequences}")
print(f"  Slice axis/idx : axis={slice_axis}, idx={slice_idx}")
print(f"  Image size     : {H}x{W}")
print()

try:
    for epoch in range(start_epoch, EPOCHS):
        epoch_start = time.perf_counter()

        if subsample_visits and n_visits > min_visits:
            n_use = random.randint(min_visits, n_visits)
            indices = sorted(random.sample(range(n_visits), n_use))
            sub_frames = [frames[i] for i in indices]
            sub_lt = torch.tensor(indices, dtype=torch.long)
        else:
            sub_frames = frames
            sub_lt = lt_seq

        loss = model.train_step(sub_frames, sub_lt)
        loss_history.append(loss)

        epoch_elapsed = time.perf_counter() - epoch_start

        if epoch % log_every == 0 or epoch == EPOCHS - 1:
            avg_recent = np.mean(loss_history[-50:])
            print(f"  Epoch {epoch:4d}/{EPOCHS}  "
                f"loss={loss:.4f}  avg50={avg_recent:.4f}  "
                f"({epoch_elapsed:.2f}s/epoch)")

        if loss < best_loss or epoch % save_every == 0:
            if loss < best_loss:
                best_loss = loss
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'best_loss': best_loss,
                'args': {""}
            }, ckpt_path)
except KeyboardInterrupt:
    print("\nTraining interrupted. Saving current model...")
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'best_loss': best_loss,
        'args': {""}
    }, ckpt_path)
        
total_elapsed = time.perf_counter() - total_start
print(f"\nTraining complete in {total_elapsed:.1f}s")
print(f"Best loss: {best_loss:.4f}")

plt.figure(figsize=(8, 3))
plt.plot(loss_history, linewidth=0.8, alpha=0.6, label='per-epoch loss')
if len(loss_history) >= 10:
    smooth = pd.Series(loss_history).rolling(10).mean()
    plt.plot(smooth, linewidth=2, label='10-epoch avg')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('RDDIM training loss — longitudinal MRI')
plt.legend()
plt.tight_layout()
loss_plot_path = output_dir / 'loss_curve.png'
loss_plot_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(loss_plot_path, dpi=150)
plt.close()
print(f"Saved loss curve → {loss_plot_path}")

print("\nSampling after training...")
sample_and_save(model, frames, lt_seq, sequences)

Model parameters: 11,988,940

Training for 100 epochs...
  Sequences used : ['FLAIR', 'POST', 'PRE', 'T2']
  Slice axis/idx : axis=2, idx=128
  Image size     : 256x256

